In [ ]:
import os, re
import numpy as np, pandas as pd
from datetime import datetime
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, confusion_matrix
)
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr


In [ ]:
base_dir = "./Aggregated"
AGGREGATED_LABELS = os.path.join(base_dir, "AllTales-Aggregated.csv")

SENTIMENT_COL = "Sentiments-Aggregated"
MULTI_FLAG_COL = "Multi-Aggregated"
SENTIMENTS = ["Negative", "Neutral", "Positive"]

timestamp = datetime.now().strftime("%y%m%d-%H%M")
out_dir = os.path.join(base_dir, "results", f"{timestamp}-5FoldCV")
os.makedirs(out_dir, exist_ok=True)

In [ ]:

if not os.path.exists(AGGREGATED_LABELS):
    raise SystemExit(f"Label file not found: {AGGREGATED_LABELS}")

labels_df = pd.read_csv(AGGREGATED_LABELS)
labels_df = labels_df.dropna(subset=[SENTIMENT_COL])
labels_df[SENTIMENT_COL] = labels_df[SENTIMENT_COL].astype(str).str.strip().str.title()
labels_df[MULTI_FLAG_COL] = labels_df[MULTI_FLAG_COL].astype(str).str.strip().str.lower()
labels_df = labels_df[labels_df[SENTIMENT_COL].isin(SENTIMENTS)]
labels_df = labels_df[labels_df[MULTI_FLAG_COL] != "yes"]

labels_df["Story"] = (
    labels_df["Story"].astype(str)
    .apply(lambda s: re.sub(r"^\s*\d+\s*-\s*", "", s.strip()))
    .str.strip().str.title()
)

print(f"Loaded {len(labels_df)} total rows after filtering.")

# merge
labels_df["Story"] = labels_df["Story"].str.lower().str.strip()
labels_df["Segment"] = labels_df["Segment"].astype(str).str.strip()

feature_folders = {
    re.sub(r"^\s*\d+-\s*", "", f.strip()).lower(): f
    for f in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, f))
}

merged = []
for story, df_lab in labels_df.groupby("Story"):
    folder = feature_folders.get(story)
    if not folder:
        continue
    feat_path = os.path.join(base_dir, folder, "AllFront_features.csv")
    if not os.path.exists(feat_path):
        continue

    df_feat = pd.read_csv(feat_path)
    df_feat["Story"] = df_feat["Story"].astype(str).str.lower()
    df_feat["Segment"] = df_feat["Segment"].astype(str).str.strip()

    df_merged = pd.merge(df_lab, df_feat, on=["Story", "Segment"], how="inner")
    if not df_merged.empty:
        merged.append(df_merged)

if not merged:
    raise SystemExit("No matching Story found.")

df_all = pd.concat(merged, ignore_index=True)
print(f"Total merged rows: {len(df_all)}")


In [ ]:
# Feature Selection
drop_cols = [
    "Story","Tale","Segment",SENTIMENT_COL,"text_original","Multi","Sentiments-Perplexity","Multi-Perplexity",
    "Sentiments-GPT5","Multi-GPT5","Sentiments-Mistral","Multi-Mistral",
    "Sentiments-GPTOSS20B","Multi-GPTOSS20B","Sentiment","Multi-Aggregated"
]
feat_cols = [c for c in df_all.columns if c not in drop_cols and np.issubdtype(df_all[c].dtype, np.number)]

features_keep = ["mouthSmileRight_mean", "mouthSmileLeft_mean", "pose_LEFT_ELBOW_z_mean", "browDownLeft_mean", "pose_LEFT_SHOULDER_z_mean", "pose_LEFT_HIP_y_velocity_std", "browOuterUpLeft_std", "mouthRight_mean", "browDownRight_mean", "pose_RIGHT_HIP_y_acceleration_std", "mouthUpperUpRight_mean", "mouthUpperUpLeft_mean", "pose_RIGHT_HIP_y_velocity_std", "pose_RIGHT_ELBOW_y_std", "mouthDimpleRight_std", "pose_RIGHT_HIP_z_mean", "browOuterUpLeft_mean", "mouthSmileRight_std", "pose_NOSE_z_mean", "left_hand_WRIST_y_mean", "browDownLeft_std", "mouthLowerDownLeft_mean", "jawRight_mean", "pose_LEFT_SHOULDER_y_velocity_std", "pose_LEFT_HIP_y_std", "pose_RIGHT_HIP_y_std", "mouthDimpleRight_mean", "mouthShrugLower_mean", "torso_yaw_mean", "right_hand_WRIST_x_acceleration_std", "pose_LEFT_ELBOW_y_std", "eyeLookInLeft_peaks_per_s", "pose_RIGHT_HIP_z_velocity_mean", "eyeSquintLeft_mean", "mouthLeft_peaks_per_s", "pose_RIGHT_HIP_y_mean", "pose_LEFT_HIP_z_std", "mouthRight_std", "mouthFrownLeft_std", "pose_LEFT_HIP_y_acceleration_std", "head_yaw_deg_mean", "mouthRollUpper_mean", "dist_right_wrist_to_left_shoulder_avg", "mouthLowerDownRight_peaks_per_s", "jawRight_std", "mouthLeft_std", "pose_RIGHT_SHOULDER_z_velocity_std", "pose_RIGHT_HIP_z_acceleration_std", "pose_RIGHT_SHOULDER_y_mean", "pose_LEFT_SHOULDER_z_std", "browOuterUpRight_std", "noseSneerRight_std", "torso_roll_mean", "mouthFrownRight_mean", "pose_LEFT_ELBOW_y_mean", "browOuterUpRight_mean", "dist_left_wrist_to_left_shoulder_peaks_per_s", "pose_RIGHT_ELBOW_y_mean", "left_arm_angle_mean", "head_pitch_deg_mean", "mouthShrugLower_std", "mouthClose_mean", "mouthFrownLeft_mean", "browInnerUp_std", "left_hand_WRIST_x_acceleration_std", "dist_right_wrist_to_nose_avg", "mouthPucker_std", "mouthPressLeft_peaks_per_s", "R_WRIST_accum_dist_avg", "pose_RIGHT_ELBOW_y_acceleration_std", "eyeLookDownRight_mean", "pose_LEFT_ELBOW_x_mean", "dist_elbows_lr_avg", "pose_NOSE_x_mean", "eyeLookDownLeft_mean", "pose_LEFT_SHOULDER_x_acceleration_mean", "pose_LEFT_HIP_y_mean", "mouthLowerDownLeft_std", "L_SHOULDER_accum_dist_avg", "pose_LEFT_HIP_z_velocity_mean", "pose_RIGHT_HIP_z_std", "eyeBlinkLeft_std", "pose_LEFT_SHOULDER_y_acceleration_mean", "pose_RIGHT_SHOULDER_z_mean", "eyeBlinkRight_std", "right_hand_WRIST_x_velocity_std", "eyeWideRight_mean", "right_hand_WRIST_y_mean", "cheekSquintRight_mean", "pose_LEFT_ELBOW_y_peaks_per_s", "mouthUpperUpRight_std", "pose_RIGHT_SHOULDER_x_std", "pose_NOSE_z_velocity_std", "mouthPressRight_std", "right_hand_WRIST_z_velocity_std"]
feat_cols = [c for c in df_all.columns if c in set(features_keep)]


label_to_id = {lbl:i for i,lbl in enumerate(SENTIMENTS)}
id_to_label = {v:k for k,v in label_to_id.items()}
y = df_all[SENTIMENT_COL].map(label_to_id).values
X = df_all[feat_cols].copy()

BASE_PARAMS = {
    "objective": "multi:softprob",
    "num_class": len(SENTIMENTS),
    "tree_method": "hist",
    "device": "cuda",
    "predictor": "gpu_predictor",
    "n_estimators": 1500,
    "eval_metric": ["mlogloss", "merror"],
    "verbosity": 0,
    "random_state": 42,
    "scale_pos_weight": 0.9,
    "alpha": 0.6,
    "colsample_bytree": 0.75,
    "eta": 0.06,
    "gamma": 0.15,
    "lambda": 2.0,
    "max_depth": 5,
    "min_child_weight": 1.5,
    "subsample": 0.85
}

# STRATIFIED 5-FOLD CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
feature_importances = []
all_predictions = []


for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    clf = xgb.XGBClassifier(**BASE_PARAMS)
    clf.fit(X_train, y_train, verbose=False)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)
    
    ordinal_map = {0: -1, 1: 0, 2: 1}
    # --- Pearson correlation per fold ---
    y_test_ord = pd.Series(y_test).map(ordinal_map)
    y_pred_ord_fold = pd.Series(y_pred).map(ordinal_map)

    r_fold, p_fold = pearsonr(y_test_ord, y_pred_ord_fold)
    print(f"Fold {fold} Pearson correlation: r={r_fold:.4f}, p={p_fold:.4f}")

    # Save predictions for this fold
    pred_df = pd.DataFrame({
        "Fold": fold,
        "Index": test_idx,
        "True_Label": [id_to_label[i] for i in y_test],
        "Pred_Label": [id_to_label[i] for i in y_pred]
    })

    for i, lbl in id_to_label.items():
        pred_df[f"Prob_{lbl}"] = y_proba[:, i]

    all_predictions.append(pred_df)

    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)

    # Per-class precision/recall/F1
    per_class = precision_recall_fscore_support(y_test, y_pred, labels=[0,1,2], zero_division=0)
    per_class_df = pd.DataFrame({
        "Class": [id_to_label[i] for i in [0,1,2]],
        "Precision": per_class[0],
        "Recall": per_class[1],
        "F1": per_class[2]
    })
    per_class_df.to_csv(os.path.join(out_dir, f"fold{fold}_perclass.csv"), index=False)

    cm = confusion_matrix(y_test, y_pred, normalize="true")
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt=".2f", xticklabels=SENTIMENTS, yticklabels=SENTIMENTS, cmap="viridis")
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"ConfMat_Fold{fold}.pdf"), dpi=300)
    plt.close()

    feat_imp = pd.Series(clf.feature_importances_, index=feat_cols)
    feat_imp = feat_imp.sort_values(ascending=False)
    feature_importances.append(feat_imp)
    
    # Save feature importances for this fold
    feat_imp.to_csv(os.path.join(out_dir, f"fold{fold}_feature_importances.csv"))

    print(f"Fold {fold}: Acc={acc:.3f}, F1={f1_m:.3f}, BalAcc={bal_acc:.3f}")

    fold_results.append({
        "Fold": fold,
        "Accuracy": acc,
        "Balanced_Accuracy": bal_acc,
        "Macro_Precision": p_m,
        "Macro_Recall": r_m,
        "Macro_F1": f1_m,
        "Weighted_Precision": p_w,
        "Weighted_Recall": r_w,
        "Weighted_F1": f1_w,
        "Recall_Negative": per_class[1][0],
        "Recall_Neutral": per_class[1][1],
        "Recall_Positive": per_class[1][2],
        "Pearson_r": r_fold,
        "Pearson_p": p_fold,
        **BASE_PARAMS
    })

fold_df = pd.DataFrame(fold_results)
fold_df.to_csv(os.path.join(out_dir, f"{timestamp}-5Fold_results.csv"), 
index=False)

# Save predictions
predictions_df = pd.concat(all_predictions, ignore_index=True)

predictions_df.to_csv(
    os.path.join(out_dir, f"{timestamp}-5Fold_predictions.csv"),
    index=False
)

# Overall confusion matrix
cm_all = confusion_matrix(
    predictions_df["True_Label"],
    predictions_df["Pred_Label"],
    labels=SENTIMENTS,
    normalize="true"
)

plt.figure(figsize=(5,4))
sns.heatmap(cm_all, annot=True, fmt=".2f",
            xticklabels=SENTIMENTS,
            yticklabels=SENTIMENTS,
            cmap="viridis")
plt.title("Confusion Matrix - 5 Fold CV (Overall)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, f"{timestamp}-ConfMat_Overall.pdf"), dpi=300)
plt.close()

In [ ]:
# Feature Importance
imp_df = pd.concat(feature_importances, axis=1).fillna(0.0)
imp_mean = imp_df.mean(axis=1).sort_values(ascending=False)
imp_std = imp_df.std(axis=1)
imp_std = imp_std[imp_mean.index]
imp_mean.to_csv(os.path.join(out_dir, f"{timestamp}-FeatureImportances_All.csv"))

# Top 20 plot
plt.figure(figsize=(6,4))
sns.barplot(x=imp_mean.head(20).values, 
            y=imp_mean.head(20).index,
            xerr=imp_std.head(20).values,
            capsize=0.2,
            color="skyblue")

plt.title("Top 20 Feature Importances", fontsize=14)
plt.xlabel("Importance mean and std", fontsize=11)
plt.ylabel("")  # remove y-axis label
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, f"{timestamp}-Top20Features.pdf"), dpi=300)
plt.close()

# Top 30 plot
plt.figure(figsize=(6,4))
sns.barplot(x=imp_mean.head(30).values, 
            y=imp_mean.head(30).index,
            xerr=imp_std.head(30).values,
            capsize=0.2,
            color="skyblue")

plt.title("Top 30 Feature Importances", fontsize=14)
plt.xlabel("Importance mean and std", fontsize=11); 
plt.ylabel("")  # remove y-axis label
plt.xticks(fontsize=9)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, f"{timestamp}-Top30Features.pdf"), dpi=300)
plt.close()

# summary & plot
summary = fold_df.describe().T[["mean","std"]]
summary.to_csv(os.path.join(out_dir, f"{timestamp}-5Fold_summary.csv"))
print("\n=== Cross-Validation Summary ===")
print(summary)

# Metrics plot
metrics = ["Accuracy","Balanced_Accuracy","Macro_F1","Weighted_Precision","Weighted_Recall"]
plt.figure(figsize=(8,4))
for m in metrics:
    plt.plot(fold_df["Fold"], fold_df[m], marker="o", label=m)
plt.ylim(0,1)  # start y-axis from 0
plt.xlabel("Fold", fontsize=12); plt.ylabel("Score", fontsize=12)
plt.title("5-Fold Cross-Validation Metrics", fontsize=14)
plt.legend(); plt.tight_layout()
plt.xticks(sorted(fold_df["Fold"].unique()))
plt.savefig(os.path.join(out_dir, f"{timestamp}-CV_Metrics.pdf"), dpi=300)
plt.close()

# Distribution plot
plt.figure(figsize=(6,4))
sns.boxplot(data=fold_df[["Accuracy","Balanced_Accuracy","Macro_F1","Weighted_Precision","Weighted_Recall"]])
plt.ylim(0,1)
plt.title("Metric Distribution Across Folds", fontsize=12)
plt.xticks(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, f"{timestamp}-Metric_Distribution.pdf"), dpi=300)
plt.close()

print(f"\nFinished. Results and plots saved in: {out_dir}")